# Disponibilizar a análise e preparar a sua reprodução
**Fase 6 — Disponibilização | CRISP-DM**

Já avaliámos os resultados. Agora vamos transformar essa análise num produto que se consegue consultar e reproduzir: **uma página local de resultados, documentação e um pacote do projeto**.

A página permite escolher mês e região, ordenar unidades e consultar a evolução, os erros de previsão e as conclusões das hipóteses. O objetivo é facilitar a leitura dos dados e deixar os limites visíveis.

Nesta versão, a disponibilização é local. Não vamos fazer uma nova previsão para julho de 2026 nem ligar os modelos antigos ao indicador novo. A escolha do `04` e a avaliação do `05` são preservadas.

| Resultado | Onde fica |
|---|---|
| Página de resultados, autónoma | `docs/06_resultados_sns.html` |
| Proposta de README | `docs/06_README_proposto.md` |
| Proposta de dependências | `docs/06_requirements_propostos.txt` |
| Decisões e atualização | `docs/06_disponibilizacao_e_atualizacao.md` |
| Inventário dos ficheiros essenciais | `data/analytics/06_inventario_essencial.csv` |
| Registo da versão disponibilizada | `docs/06_disponibilizacao_manifesto.json` |
| Pacote local do projeto | `data/output/Projeto_SNS.zip`, quando os essenciais estiverem presentes |

**Execução:** colocar este notebook em `notebooks/`, usar o mesmo kernel com `pandas` e `matplotlib`, e executar com **Run All**. Não há novas bibliotecas, nova recolha ou treino de modelos nesta fase.

## 1. Confirmar a avaliação e os ficheiros que lhe deram origem

Vamos ler o resumo do `05` e verificar as assinaturas das suas entradas. Os caminhos guardados em Windows e Linux são normalizados antes da comparação.

Confirmamos também que o MAE do resumo corresponde às previsões guardadas e que a seleção continua a ser a do `04`. Se uma entrada mudou, paramos para não apresentar uma página com resultados de execuções diferentes.

In [1]:
from pathlib import Path, PureWindowsPath
import base64
import hashlib
import json
import os
import platform
import zipfile
from html import escape
from importlib.metadata import version, PackageNotFoundError
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

atual = Path.cwd().resolve()
raiz = next((p for p in [atual, *atual.parents]
             if (p / "notebooks").is_dir() and (p / "data").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("Abrir a pasta do projeto, com notebooks/ e data/.")
PASTA_OUTPUT = raiz / "data" / "output"
PASTA_ANALYTICS = raiz / "data" / "analytics"
PASTA_DOCS = raiz / "docs"
for pasta in [PASTA_OUTPUT, PASTA_ANALYTICS, PASTA_DOCS]:
    pasta.mkdir(parents=True, exist_ok=True)

ficheiro_resumo = PASTA_OUTPUT / "sns_avaliacao_resumo.json"
if not ficheiro_resumo.is_file():
    raise FileNotFoundError("Falta sns_avaliacao_resumo.json. Executar o 05 antes do 06.")
resumo = json.loads(ficheiro_resumo.read_text(encoding="utf-8"))

def sha256(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

# Normalizar também os registos de execuções antigas em Windows.
assinaturas_avaliadas = {PureWindowsPath(nome).as_posix(): valor
                        for nome, valor in resumo["input_sha256"].items()}
entradas = [ficheiro_resumo]
for nome, assinatura in assinaturas_avaliadas.items():
    caminho = raiz / nome
    if not caminho.is_file():
        raise FileNotFoundError(f"Falta uma entrada avaliada no 05: {nome}.")
    if sha256(caminho) != assinatura:
        raise ValueError(f"O ficheiro mudou desde a avaliação: {nome}. Reexecutar as fases dependentes.")
    entradas.append(caminho)
assinaturas_antes = {p.relative_to(raiz).as_posix(): sha256(p) for p in entradas}
principal = pd.read_csv(PASTA_OUTPUT / "sns_csp_principal.csv", parse_dates=["mes"])
regional = pd.read_csv(PASTA_OUTPUT / "sns_csp_regional.csv", parse_dates=["mes"])
previsoes = pd.read_csv(PASTA_OUTPUT / "sns_previsoes_teste.csv", parse_dates=["mes", "mes_alvo"])
execucao_04 = json.loads((raiz / "data/models/04_execucao_modelacao.json").read_text(encoding="utf-8"))
metricas = pd.DataFrame(resumo["metricas_teste"])
INICIO = principal["mes"].min()
FIM = principal["mes"].max()
selecionado = resumo["selecionado_no_04"]
ROTULOS = {"ultimo_valor": "Último valor", "regressao_linear": "Regressão linear",
           "random_forest_a": "Random Forest A", "random_forest_b": "Random Forest B"}
checks = []
for r in metricas.itertuples():
    mae_csv = float((previsoes[f"prev_{r.abordagem}_pct"] - previsoes["observado_pct"]).abs().mean())
    checks.append({"verificacao": f"MAE do resumo coincide com as previsões: {r.abordagem}",
                   "passou": abs(mae_csv - r.mae_pp) <= 1e-10})
checks += [
    {"verificacao": "Seleção mantida desde o 04", "passou": resumo["selecao_mantida"] and selecionado == execucao_04["selecionado"]},
    {"verificacao": "Período do resumo coincide com o principal", "passou": INICIO.strftime("%Y-%m") == resumo["periodo_principal"]["inicio"] and FIM.strftime("%Y-%m") == resumo["periodo_principal"]["fim"]},
    {"verificacao": "Principal sem chaves duplicadas", "passou": not principal.duplicated(["regiao_saude", "unidade_fonte", "mes"]).any()},
]
validacoes = pd.DataFrame(checks)
validacoes.to_csv(PASTA_ANALYTICS / "06_validacoes_entrada.csv", index=False, encoding="utf-8-sig")
display(validacoes)
if not validacoes["passou"].all():
    raise ValueError("Há incoerências nas entradas. Rever o 05 antes de disponibilizar.")
print("Avaliação conferida. Produto histórico:", INICIO.strftime("%Y-%m"), "a", FIM.strftime("%Y-%m"))


,verificacao,passou
0,MAE do resumo coincide com as previsões: ultim...,True
1,MAE do resumo coincide com as previsões: regre...,True
2,MAE do resumo coincide com as previsões: rando...,True
3,Seleção mantida desde o 04,True
4,Período do resumo coincide com o principal,True
5,Principal sem chaves duplicadas,True


Avaliação conferida. Produto histórico: 2024-01 a 2026-06


## 2. Preparar os gráficos do produto

A página vai mostrar a evolução regional e a comparação dos erros no teste. Os gráficos são gerados das entradas conferidas, sem depender de ficheiros de imagem copiados de outra execução.

As percentagens regionais já foram calculadas pelos totais no `03`. O erro de previsão dá o mesmo peso a cada unidade/mês, como definido na avaliação. Não recalculamos nem escolhemos modelos nesta fase.

In [2]:
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

def guardar_grafico(fig, nome):
    fig.tight_layout()
    caminho = PASTA_ANALYTICS / nome
    fig.savefig(caminho, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return caminho

fig, ax = plt.subplots(figsize=(10, 4.5))
for regiao, dados in regional.groupby("regiao_saude"):
    dados = dados.sort_values("mes")
    ax.plot(dados["mes"], dados["sem_medico_pct"], label=regiao, linewidth=2)
ax.set(title="Evolução por região de saúde", xlabel="Mês de referência", ylabel="Inscritos sem médico (%)")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.grid(axis="y", alpha=0.18)
grafico_regional = guardar_grafico(fig, "06_evolucao_regional.png")

fig, ax = plt.subplots(figsize=(9, 4))
tabela = metricas.sort_values("mae_pp", ascending=False)
ax.barh([ROTULOS.get(n, n) for n in tabela["abordagem"]], tabela["mae_pp"], color="#176a8b")
ax.set(title="Previsões a um mês — teste de janeiro a junho de 2026", xlabel="MAE (pontos percentuais)")
ax.grid(axis="x", alpha=0.18)
grafico_modelos = guardar_grafico(fig, "06_comparacao_modelos.png")
print("Gráficos preparados para a página de resultados.")


Gráficos preparados para a página de resultados.


## 3. Gerar a página local de resultados

O ficheiro HTML contém os dados necessários à tabela e as imagens dos gráficos. Abre com duplo clique num navegador e funciona sem servidor ou ligação à API.

Os filtros alteram a tabela e os totais do mês/região selecionados. Os gráficos mantêm a visão do período completo. A categoria por opção é apresentada separadamente e a proporção sem médico é calculada a partir dos totais visíveis.

O texto da fonte é tratado como texto, para que nomes de unidades não sejam interpretados como código na página. As perguntas, hipóteses, decisões e limites vêm do resumo avaliado no `05`.

In [3]:
def imagem_embebida(caminho):
    return "data:image/png;base64," + base64.b64encode(caminho.read_bytes()).decode("ascii")

campos = ["mes", "regiao_saude", "unidade_fonte", "inscritos", "com_medico", "sem_medico",
          "sem_medico_opcao", "sem_medico_pct", "sem_medico_opcao_pct"]
dados_pagina = principal[campos].copy()
dados_pagina["mes"] = dados_pagina["mes"].dt.strftime("%Y-%m")
conteudo = {
    "linhas": dados_pagina.to_dict("records"),
    "meses": sorted(dados_pagina["mes"].unique()),
    "regioes": sorted(dados_pagina["regiao_saude"].unique()),
}
# Evitar que texto da fonte seja interpretado como HTML ou como fim de script.
payload = json.dumps(conteudo, ensure_ascii=False, allow_nan=False).replace("<", "\\u003c").replace(">", "\\u003e").replace("&", "\\u0026")

hipoteses_html = "".join(
    "<article class='hyp'><span class='tag'>" + escape(h["id"]) + "</span><h3>" + escape(h["hipotese"]) +
    "</h3><p>" + escape(h["evidencia"]) + "</p><p class='muted'>" + escape(h["avaliacao"]) + "</p></article>"
    for h in resumo["hipoteses"])
perguntas_html = "".join(
    "<details><summary>" + escape(q["id"] + " · " + q["pergunta"]) + "</summary><p>" + escape(q["resposta"]) +
    "</p><p class='muted'>" + escape(q["estado"]) + "</p></details>" for q in resumo["perguntas"])
limites_html = "".join("<li>" + escape(t) + "</li>" for t in resumo["limites"])
pendencias_html = "".join("<li>" + escape(t) + "</li>" for t in resumo["pendencias"])
tabela_metricas = metricas[["abordagem", "mae_pp", "rmse_pp", "vies_pp"]].copy()
tabela_metricas["abordagem"] = tabela_metricas["abordagem"].map(lambda s: ROTULOS.get(s, s))
tabela_metricas.columns = ["Abordagem", "MAE (pp)", "RMSE (pp)", "Viés (pp)"]
metricas_html = tabela_metricas.to_html(index=False, escape=True, border=0, classes="metrics",
    float_format=lambda v: f"{v:.3f}".replace(".", ","))

HTML_MODELO = r'''<!doctype html>
<html lang="pt-PT"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Cobertura de médico de família — análise histórica SNS</title>
<style>
:root{--ink:#142b3a;--muted:#526979;--blue:#176a8b;--bg:#f3f6f8;--line:#dbe4e9;--paper:#fff}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.6 system-ui,-apple-system,Segoe UI,sans-serif}
header{background:#102f42;color:#fff;padding:48px max(24px,calc((100vw - 1180px)/2)) 34px}
header p{max-width:780px;color:#d7e6ef}h1{font-size:clamp(27px,4vw,43px);line-height:1.15;margin:12px 0 18px;letter-spacing:-.02em}
h2{font-size:25px;line-height:1.25;margin:0 0 16px}h3{font-size:18px;line-height:1.4;margin:12px 0}
a{color:var(--blue)}header a{color:#d7e6ef}nav{display:flex;flex-wrap:wrap;gap:18px;margin-top:22px}
main{max-width:1228px;margin:auto;padding:26px 24px 48px}section{background:var(--paper);padding:28px;border:1px solid var(--line);border-radius:14px;margin-bottom:24px;scroll-margin-top:16px}
.eyebrow{font-size:13px;text-transform:uppercase;letter-spacing:.1em;color:#a5d3e7}.notice{background:#e9f2f6;border-left:4px solid var(--blue);padding:14px 18px;border-radius:0 8px 8px 0}
.muted{color:var(--muted);font-size:14px}.controls{display:grid;grid-template-columns:1fr 1.5fr 1.5fr;gap:16px;margin:22px 0}label{display:block;font-weight:600;font-size:14px}
select{display:block;width:100%;font:inherit;color:var(--ink);background:white;border:1px solid #afc2cc;border-radius:7px;padding:9px;margin-top:5px}
.cards{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin:18px 0}.card{padding:18px;background:#eef5f8;border-radius:10px}.card strong{display:block;font-size:30px;line-height:1.3}.card span{color:var(--muted);font-size:14px}
.table-wrap{overflow-x:auto;max-width:100%;margin-top:14px}table{width:100%;border-collapse:collapse;font-size:14px}th,td{text-align:right;padding:11px 12px;border-bottom:1px solid var(--line);white-space:nowrap}th{text-transform:none;background:#edf3f6;font-weight:600}th:first-child,td:first-child,th:nth-child(2),td:nth-child(2){text-align:left}td:first-child{white-space:normal;min-width:230px}tbody tr:nth-child(even){background:#fafcfd}
figure{margin:20px 0 0}figure img{display:block;width:100%;height:auto;border:1px solid var(--line);border-radius:8px}figcaption{font-size:14px;color:var(--muted);padding-top:9px}
.hyp-grid{display:grid;grid-template-columns:1fr 1fr;gap:16px}.hyp{border:1px solid var(--line);border-radius:10px;padding:20px}.hyp p{margin-bottom:0}.tag{display:inline-block;background:#e4f0f5;color:#14516b;font-size:13px;font-weight:700;border-radius:5px;padding:2px 8px}
details{border-top:1px solid var(--line);padding:14px 0}summary{cursor:pointer;font-weight:600}details p{max-width:1000px}footer{color:var(--muted);font-size:14px;padding:0 4px}
@media(max-width:720px){header{padding:30px 20px}main{padding:18px 14px}section{padding:20px}.controls,.cards,.hyp-grid{grid-template-columns:1fr}.card strong{font-size:27px}nav{gap:12px}h2{font-size:22px}}
@media print{body{background:#fff}header{background:#fff;color:#000;padding:12px}header p,header a{color:#000}nav,.controls{display:none}section{break-inside:avoid;border:none;padding:12px}main{padding:0}.table-wrap{overflow:visible}th,td{white-space:normal;font-size:10px}.cards{grid-template-columns:repeat(3,1fr)}details>p{display:block}figure{break-inside:avoid}}
</style></head><body>
<header><div class="eyebrow">Análise independente · Dados públicos do SNS</div><h1>Cobertura de médico de família</h1>
<p>Onde se concentra a falta de médico atribuído, como evoluiu a cobertura e o que acrescentou a previsão do mês seguinte.</p>
<p><strong>{{PERIODO}}</strong> · Portugal Continental · Recolha {{RECOLHA}}</p>
<nav aria-label="Secções"><a href="#cobertura">Cobertura</a><a href="#evolucao">Evolução</a><a href="#previsao">Previsão</a><a href="#hipoteses">Hipóteses</a><a href="#limites">Âmbito e limites</a></nav></header>
<main>
<section id="cobertura"><h2>Cobertura no mês de referência</h2><p>Escolhe um mês e uma região para consultar os inscritos e comparar as unidades. A categoria sem médico por opção fica separada.</p>
<div class="controls"><label>Mês<select id="mes" aria-label="Mês de referência"></select></label><label>Região de saúde<select id="regiao" aria-label="Região de saúde"></select></label><label>Ordenar unidades por<select id="ordem" aria-label="Ordenar unidades"><option value="sem_medico_pct">Percentagem sem médico</option><option value="sem_medico">Número sem médico</option><option value="inscritos">Total de inscritos</option></select></label></div>
<div class="cards" aria-live="polite"><div class="card"><strong id="inscritos">—</strong><span>Inscritos no mês e região selecionados</span></div><div class="card"><strong id="sem-medico">—</strong><span>Sem médico atribuído, excluindo a categoria por opção</span></div><div class="card"><strong id="percentagem">—</strong><span>Percentagem calculada pelos totais</span></div></div>
<p id="contexto" class="muted"></p><p id="opcao" class="muted"></p>
<div class="table-wrap"><table id="ranking"><thead><tr><th>Unidade de saúde</th><th>Região</th><th>Inscritos</th><th>Sem médico</th><th>Sem médico (%)</th><th>Por opção</th></tr></thead><tbody></tbody></table></div>
<p class="muted">São inscritos na fonte, não habitantes residentes. Cada mês é uma fotografia: não somar os meses como se fossem pessoas distintas.</p></section>
<section id="evolucao"><h2>Evolução regional</h2><p>As percentagens regionais resultam da soma das contagens a dividir pela soma dos inscritos em cada mês. Os gráficos apresentam o período completo, independentemente dos filtros da tabela.</p>
<figure><img src="{{IMAGEM_REGIONAL}}" alt="Evolução mensal da percentagem sem médico em cada região de saúde"><figcaption>Todos os registos mensais do recorte principal. A comparação de evolução por unidade usa as 38 séries completas.</figcaption></figure></section>
<section id="previsao"><h2>O histórico melhorou a previsão?</h2><div class="notice"><strong>Escolha feita na validação: {{SELECIONADO}}</strong><p>{{DECISAO}}</p></div>
<p>Teste com meses-alvo de {{TESTE_PERIODO}}, em {{TESTE_UNIDADES}} unidades e {{TESTE_LINHAS}} linhas. Cada previsão usa informação do mês de origem ou anterior. A seleção foi feita antes de avaliar este teste.</p>
<div class="table-wrap">{{METRICAS}}</div><p class="muted">MAE e RMSE em pontos percentuais. Viés negativo indica subestimação. Cada unidade/mês tem o mesmo peso nesta avaliação.</p>
<figure><img src="{{IMAGEM_MODELOS}}" alt="Comparação do erro absoluto médio das previsões no período de teste"><figcaption>Mesmas unidades e meses para as três abordagens. Esta é uma avaliação histórica do indicador antigo.</figcaption></figure></section>
<section id="hipoteses"><h2>O que mostram as quatro hipóteses</h2><div class="hyp-grid">{{HIPOTESES}}</div></section>
<section id="perguntas"><h2>Respostas às perguntas do projeto</h2>{{PERGUNTAS}}</section>
<section id="limites"><h2>Âmbito, limites e definições pendentes</h2><div class="notice">O recorte começa em janeiro de 2024 após a reorganização em ULS e termina em junho de 2026 antes da quebra de definição do indicador. O histórico bruto desde 2016 fica preservado.</div>
<ul>{{LIMITES}}</ul><h3>Definições ainda por confirmar</h3><ul>{{PENDENCIAS}}</ul>
<p>A análise apoia a leitura dos indicadores. Não determina contratação ou distribuição de médicos, nem constitui uma previsão atual do indicador publicado a partir de julho de 2026.</p>
<h3>Fonte e continuidade da série</h3><p><a href="https://transparencia.sns.gov.pt/explore/assets/utentes-inscritos-em-cuidados-de-saude-primarios/">Indicador histórico no Portal da Transparência do SNS</a> · <a href="https://transparencia.sns.gov.pt/api/explore/v2.1/catalog/datasets/utentes-inscritos-em-cuidados-de-saude-primarios">Metadados e API oficial</a> · <a href="https://www.ers.pt/pt/atividade/supervisao/selecionar/alertas-de-supervisao/2024/alertas/alerta-de-supervis%C3%A3o-n-%C2%BA-1-2024/">ERS — reorganização em ULS</a></p></section>
<footer>Resultados calculados a partir da recolha identificada. A página funciona localmente; os dados e gráficos ficam incluídos neste ficheiro. Para reproduzir a análise, consultar o README e os seis notebooks.</footer>
</main><script type="application/json" id="dados">{{PAYLOAD}}</script>
<script>
const dados=JSON.parse(document.getElementById('dados').textContent);
const mes=document.getElementById('mes'), regiao=document.getElementById('regiao'), ordem=document.getElementById('ordem');
const numero=new Intl.NumberFormat('pt-PT'), percent=new Intl.NumberFormat('pt-PT',{minimumFractionDigits:2,maximumFractionDigits:2});
for(const value of [...dados.meses].reverse())mes.add(new Option(value,value));
regiao.add(new Option('Todas as regiões',''));for(const value of dados.regioes)regiao.add(new Option(value,value));
function atualizar(){
 const linhas=dados.linhas.filter(r=>r.mes===mes.value&&(!regiao.value||r.regiao_saude===regiao.value)).sort((a,b)=>b[ordem.value]-a[ordem.value]||a.unidade_fonte.localeCompare(b.unidade_fonte,'pt'));
 const total=linhas.reduce((s,r)=>s+r.inscritos,0), sem=linhas.reduce((s,r)=>s+r.sem_medico,0), opcao=linhas.reduce((s,r)=>s+r.sem_medico_opcao,0);
 document.getElementById('inscritos').textContent=numero.format(total);document.getElementById('sem-medico').textContent=numero.format(sem);document.getElementById('percentagem').textContent=total?percent.format(100*sem/total)+'%':'—';
 document.getElementById('contexto').textContent=mes.value+' · '+(regiao.value||'Portugal Continental')+' · '+linhas.length+' unidades publicadas no mês';
 document.getElementById('opcao').textContent='Categoria por opção, apresentada separadamente: '+numero.format(opcao)+' inscritos'+(total?' ('+percent.format(100*opcao/total)+'%).':'.');
 const tbody=document.querySelector('#ranking tbody');tbody.replaceChildren();
 for(const r of linhas){const tr=document.createElement('tr');for(const value of [r.unidade_fonte.replace('CSP da ULS ',''),r.regiao_saude,numero.format(r.inscritos),numero.format(r.sem_medico),percent.format(r.sem_medico_pct)+'%',numero.format(r.sem_medico_opcao)]){const td=document.createElement('td');td.textContent=value;tr.appendChild(td);}tbody.appendChild(tr);}
}
for(const select of [mes,regiao,ordem])select.addEventListener('change',atualizar);atualizar();
</script></body></html>'''

substituicoes = {
    "PERIODO": f"{INICIO:%Y-%m} a {FIM:%Y-%m}", "RECOLHA": resumo["recolhido_em_utc"].split("T")[0],
    "SELECIONADO": ROTULOS.get(selecionado, selecionado), "DECISAO": resumo["decisao_previsao"],
    "TESTE_PERIODO": resumo["teste"]["primeiro_mes_alvo"] + " a " + resumo["teste"]["ultimo_mes_alvo"],
    "TESTE_UNIDADES": str(resumo["teste"]["unidades"]), "TESTE_LINHAS": str(resumo["teste"]["linhas"]),
}
html = HTML_MODELO
for chave, texto in substituicoes.items():
    html = html.replace("{{" + chave + "}}", escape(texto))
for chave, texto in {"IMAGEM_REGIONAL": imagem_embebida(grafico_regional), "IMAGEM_MODELOS": imagem_embebida(grafico_modelos),
                     "METRICAS": metricas_html, "HIPOTESES": hipoteses_html, "PERGUNTAS": perguntas_html,
                     "LIMITES": limites_html, "PENDENCIAS": pendencias_html, "PAYLOAD": payload}.items():
    html = html.replace("{{" + chave + "}}", texto)
if "{{" in html:
    raise ValueError("A página ficou com elementos por preencher.")
ficheiro_html = PASTA_DOCS / "06_resultados_sns.html"
ficheiro_html.write_text(html, encoding="utf-8")
print("Página criada: docs/06_resultados_sns.html")
print("Abrir com duplo clique no explorador de ficheiros; não precisa de servidor.")


Página criada: docs/06_resultados_sns.html
Abrir com duplo clique no explorador de ficheiros; não precisa de servidor.


## 4. Documentar a execução e a atualização

O README explica o objetivo, a fonte, o recorte, a organização dos ficheiros, a ordem de execução e os limites. As dependências propostas usam as versões registadas na modelação e a versão do `matplotlib` que gera esta página. O kernel Jupyter é acrescentado para executar os notebooks.

Se `README.md` ou `requirements.txt` já existirem, são preservados e as propostas ficam em `docs/`. Se faltarem, são criados na raiz. Antes de fechar o projeto, podemos integrar as propostas no README existente e rever o `.gitignore`.

Criar um ambiente novo é uma instrução para quem quiser reproduzir o projeto; não precisas de repetir a instalação se o teu kernel já funciona. Os ficheiros brutos e os resultados anteriores são apenas lidos.

In [4]:
versoes_04 = execucao_04["versoes"]
dependencias = []
for biblioteca in ["pandas", "numpy", "scikit-learn", "joblib"]:
    dependencias.append(f"{biblioteca}=={versoes_04[biblioteca]}")
dependencias += [f"matplotlib=={version('matplotlib')}", "ipykernel>=6"]
requirements = "# Versões registadas na execução do 04; matplotlib usado no 06.\n"
requirements += f"# Python da modelação: {versoes_04['python']}.\n"
requirements += "\n".join(dependencias) + "\n"
proposta_requirements = PASTA_DOCS / "06_requirements_propostos.txt"
proposta_requirements.write_text(requirements, encoding="utf-8")
ficheiro_requirements = raiz / "requirements.txt"
if not ficheiro_requirements.exists():
    ficheiro_requirements.write_text(requirements, encoding="utf-8")
    print("requirements.txt criado na raiz.")
else:
    print("requirements.txt existente preservado; proposta guardada em docs/06_requirements_propostos.txt.")

mae_ref = next(m["mae_pp"] for m in resumo["metricas_teste"] if m["abordagem"] == "ultimo_valor")
mae_escolhido = next(m["mae_pp"] for m in resumo["metricas_teste"] if m["abordagem"] == selecionado)
readme = f"""# Cobertura de médico de família — SNS

Análise independente dos inscritos nos cuidados de saúde primários em Portugal Continental, por unidade e região de saúde. O projeto compara o número e a percentagem de inscritos sem médico, acompanha a evolução e avalia previsões históricas a um mês.

## Consultar os resultados

Abrir [a página de resultados](docs/06_resultados_sns.html) num navegador. Funciona localmente, sem servidor: permite escolher mês, região e ordenação dos rankings. A página inclui os dados da tabela e os gráficos.

Período principal: **{INICIO:%Y-%m} a {FIM:%Y-%m}**. Recolha: {resumo['recolhido_em_utc']} (UTC). Há 39 registos mensais no principal; a evolução por unidade e a previsão usam 38 séries completas.

Na avaliação desta execução, a abordagem escolhida na validação foi **{ROTULOS.get(selecionado, selecionado)}**. O seu MAE no teste foi {mae_escolhido:.3f} pontos percentuais; a referência de repetir o último valor teve {mae_ref:.3f} pp. {resumo['decisao_previsao']}

## Fonte e período

- [Indicador histórico do Portal da Transparência do SNS](https://transparencia.sns.gov.pt/explore/assets/utentes-inscritos-em-cuidados-de-saude-primarios/).
- [Metadados e API](https://transparencia.sns.gov.pt/api/explore/v2.1/catalog/datasets/utentes-inscritos-em-cuidados-de-saude-primarios).
- Histórico bruto preservado desde 2016. Recolha pela API oficial; sem scraping nesta versão.
- Recorte desde janeiro de 2024 devido à reorganização em ULS, com alterações territoriais. [Enquadramento da ERS](https://www.ers.pt/pt/atividade/supervisao/selecionar/alertas-de-supervisao/2024/alertas/alerta-de-supervis%C3%A3o-n-%C2%BA-1-2024/).
- Fim em junho de 2026 devido à mudança de regras e definição do indicador a partir de julho. Não juntar os dois indicadores como uma série contínua.

## Organização

| Local | Conteúdo |
|---|---|
| notebooks/ | Um notebook por fase CRISP-DM, com decisões e resultados. |
| data/input/ | Histórico bruto e metadados, preservados. |
| data/output/ | Dados preparados, previsões e resumos. |
| data/analytics/ | Verificações, tabelas, gráficos e métricas. |
| data/models/ | Pipelines guardadas e registo da seleção/modelação. |
| src/ | Funções reutilizáveis de recolha. |
| docs/ | Decisões, instruções e página de resultados. |
| media/ | Imagens de apoio, quando existirem. |

## Preparar um ambiente novo

O Python usado na modelação foi {versoes_04['python']}. As versões das bibliotecas estão em `data/models/04_execucao_modelacao.json`. Usar preferencialmente a mesma versão de Python e as dependências registadas para reproduzir o treino; podem existir pequenas diferenças entre plataformas e versões.

Na raiz do projeto, em CMD do Windows:

```cmd
python -m venv .venv
.venv\\Scripts\\python -m pip install -r requirements.txt
```

No VS Code, selecionar esse ambiente em **Select Kernel → Python Environments**. É necessário ter as extensões Python e Jupyter disponíveis. Não é preciso recriar o ambiente se já tens um kernel funcional com as bibliotecas.

## Executar

Executar os notebooks por ordem, com Run All:

1. [01 — Entendimento do negócio](notebooks/01_business_understanding.ipynb).
2. [02 — Entendimento dos dados](notebooks/02_data_understanding.ipynb).
3. [03 — Preparação](notebooks/03_data_preparation.ipynb).
4. [04 — Modelação](notebooks/04_modeling.ipynb).
5. [05 — Avaliação](notebooks/05_evaluation.ipynb).
6. [06 — Disponibilização](notebooks/06_deployment.ipynb).

O 02 reutiliza a recolha existente depois de verificar as assinaturas. Os restantes leem os ficheiros guardados; o 04 treina modelos. Os resultados gerados podem ser atualizados ao executar novamente cada fase. Se mudares uma fase, executar novamente as fases que dependem dela.

O 06 guarda a página em `docs/06_resultados_sns.html` e prepara `data/output/Projeto_SNS.zip` quando os ficheiros essenciais estiverem presentes. O pacote reúne a versão local do projeto e exclui ambientes virtuais, caches e o próprio pacote.

## Interpretação e limites

- Inscritos não são população residente; não extrapolar o indicador para todos os habitantes.
- A categoria sem médico por opção fica separada de sem médico atribuído.
- Cada mês é uma fotografia: não somar pessoas entre meses. Percentagens regionais usam os totais, não a média simples das unidades.
- A correspondência territorial dos dois nomes da Póvoa/Vila do Conde não foi confirmada. Ambos ficam no principal, mas fora das séries completas de evolução e previsão.
- A janela exata das taxas de consultas de um ano continua pendente. A Q5 não foi avaliada nesta versão.
- Meses e unidades não são observações independentes; faixas de dimensão são aproximadas. As associações não identificam causas.
- O teste usa a recolha atual, que pode incluir revisões. A escolha foi feita pela validação temporal; o teste não serviu para ajustar os modelos.
- A página apresenta uma análise histórica do indicador antigo. Não constitui previsão operacional do novo indicador nem determina distribuição de médicos.

## Atualização

Se a fonte mudar, recolher para uma pasta separada e confirmar esquema, definições e territórios antes de atualizar a análise. Não substituir os brutos guardados sem preservar a recolha anterior. Para o indicador novo, definir um projeto/série próprios; não reaproveitar o teste antigo como avaliação independente depois de ajustar modelos com os seus resultados.
"""
proposta_readme = PASTA_DOCS / "06_README_proposto.md"
proposta_readme.write_text(readme, encoding="utf-8")
ficheiro_readme = raiz / "README.md"
if not ficheiro_readme.exists():
    ficheiro_readme.write_text(readme, encoding="utf-8")
    print("README.md criado na raiz.")
else:
    print("README.md existente preservado; proposta completa guardada em docs/06_README_proposto.md.")

operacao = f"""# Fase 6 — Disponibilização e atualização

Produto: página local de resultados históricos, notebooks e pacote reproduzível. Período: {INICIO:%Y-%m} a {FIM:%Y-%m}. Recolha: {resumo['recolhido_em_utc']}.

## Decisões

- Página HTML autónoma, com filtros por mês/região e ordenação dos rankings. Não exige servidor ou nova recolha para consultar.
- Não faz novas previsões nem altera a seleção do 04. Disponibiliza a avaliação histórica e os limites conhecidos.
- Dados e gráficos são gerados a partir das entradas conferidas do 05. Originais, modelos e métricas anteriores são apenas lidos.
- README e requirements existentes são preservados; propostas ficam em docs para revisão. Quando não existem, são criados na raiz.
- Pacote local reúne notebooks, dados, src, docs e media, com README/requirements. Ambientes e caches ficam de fora; o .gitignore, quando existir, é incluído. A revisão do .gitignore fica para o fecho do projeto.
- A criação desta página não implica publicação externa. Qualquer atualização tem de indicar a recolha, as definições e o período.

## Manutenção

1. Nova recolha numa pasta separada; preservar a anterior.
2. Conferir esquema, conceitos, categorias e territórios.
3. Reexecutar preparação, modelação e avaliação dependentes da mudança, com validação temporal apropriada.
4. Gerar novamente a página e o pacote; verificar os resultados e limites antes de os disponibilizar.

A série antiga termina em junho de 2026. Uma análise atual teria de usar a definição nova separadamente, com histórico suficiente e novo enquadramento. Mantêm-se pendentes a correspondência da Póvoa/Vila do Conde e a janela das consultas.
"""
(PASTA_DOCS / "06_disponibilizacao_e_atualizacao.md").write_text(operacao, encoding="utf-8")
print("Instruções e decisões de disponibilização guardadas em docs/.")


requirements.txt criado na raiz.
README.md existente preservado; proposta completa guardada em docs/06_README_proposto.md.
Instruções e decisões de disponibilização guardadas em docs/.


## 5. Verificar se o projeto está pronto para ser reunido num pacote

O inventário confere a presença dos seis notebooks, da recolha original, dos principais resultados e das instruções. A página pode estar pronta mesmo que falte um ficheiro do pacote; nesse caso, o notebook identifica o que falta e não cria um ZIP incompleto.

A presença de um ficheiro não substitui a leitura e revisão do seu conteúdo. Nesta fase, as assinaturas e métricas conferem a coerência das entradas; a revisão final inclui abrir a página, experimentar os filtros e confirmar as conclusões.

In [5]:
notebooks = ["01_business_understanding.ipynb", "02_data_understanding.ipynb", "03_data_preparation.ipynb",
             "04_modeling.ipynb", "05_evaluation.ipynb", "06_deployment.ipynb"]
essenciais = ["notebooks/" + n for n in notebooks] + [
    "README.md", "requirements.txt", "src/recolha_sns.py",
    "data/input/utentes_csp_historico_bruto.csv", "data/input/utentes_csp_metadados_api.json", "data/input/utentes_csp_recolha.json",
    "data/output/sns_csp_principal.csv", "data/output/sns_csp_modelacao.csv", "data/output/sns_preparacao_manifesto.json",
    "data/output/sns_previsoes_teste.csv", "data/output/sns_avaliacao_resumo.json",
    "data/models/04_selecao_modelo.json", "data/models/04_execucao_modelacao.json", "docs/06_resultados_sns.html",
]
inventario = pd.DataFrame([{"ficheiro": nome, "presente": (raiz / nome).is_file(),
                           "bytes": (raiz / nome).stat().st_size if (raiz / nome).is_file() else 0} for nome in essenciais])
inventario.to_csv(PASTA_ANALYTICS / "06_inventario_essencial.csv", index=False, encoding="utf-8-sig")
faltam_pacote = inventario.loc[~inventario["presente"], "ficheiro"].tolist()
if faltam_pacote:
    display(inventario.loc[~inventario["presente"]])
    print("Página criada. O pacote completo aguarda os ficheiros listados acima.")
else:
    print("Todos os", len(essenciais), "ficheiros essenciais estão presentes.")
if assinaturas_antes != {p.relative_to(raiz).as_posix(): sha256(p) for p in entradas}:
    raise ValueError("Uma entrada mudou durante a disponibilização.")
print("Entradas do 03/04/05 preservadas.")


Todos os 20 ficheiros essenciais estão presentes.
Entradas do 03/04/05 preservadas.


## 6. Guardar o registo da disponibilização e criar o pacote

O pacote inclui os ficheiros das pastas acordadas: `notebooks`, `data`, `src`, `docs` e `media`, com README e dependências. Inclui o `.gitignore` se já existir, mas não o altera.

Ambientes virtuais, caches, logs, ficheiros ocultos e outros ZIP ficam de fora. Só percorremos as pastas do projeto. O próprio pacote e o seu temporário ficam excluídos, para evitar um ZIP dentro de outro a cada execução.

**Antes do pacote final:** depois de Run All, guarda os notebooks com **Ctrl + S** e executa novamente esta última célula. A célula lê os ficheiros guardados em disco; assim o ZIP inclui as versões já executadas e guardadas. Não é necessário voltar a treinar modelos.

O registo usa caminhos com `/` para ser legível entre plataformas. Lista as assinaturas dos ficheiros reunidos, sem incluir a assinatura do próprio registo ou ZIP, para evitar uma referência circular.

In [6]:
ficheiro_manifesto = PASTA_DOCS / "06_disponibilizacao_manifesto.json"
ficheiro_pacote = PASTA_OUTPUT / "Projeto_SNS.zip"
temporario_pacote = PASTA_OUTPUT / "Projeto_SNS.zip.tmp"
IGNORAR_PASTAS = {"__pycache__", ".ipynb_checkpoints", ".git", ".venv", "venv", "env", "node_modules", ".pytest_cache", ".mypy_cache", ".ruff_cache"}
IGNORAR_FICHEIROS = {ficheiro_manifesto.resolve(), ficheiro_pacote.resolve(), temporario_pacote.resolve()}
ficheiros_pacote = []
# Percorrer só as pastas do projeto; não procurar noutras pastas do computador.
for nome_pasta in ["notebooks", "data", "src", "docs", "media"]:
    pasta = raiz / nome_pasta
    if not pasta.is_dir():
        continue
    for origem, pastas, ficheiros in os.walk(pasta):
        pastas[:] = sorted(p for p in pastas if p not in IGNORAR_PASTAS and not p.startswith("."))
        for nome in sorted(ficheiros):
            caminho = Path(origem) / nome
            if caminho.is_symlink() or caminho.resolve() in IGNORAR_FICHEIROS:
                continue
            if nome.startswith(".") or caminho.suffix.lower() in {".pyc", ".pyo", ".log", ".tmp", ".zip"}:
                continue
            ficheiros_pacote.append(caminho)
for nome in ["README.md", "requirements.txt", ".gitignore"]:
    caminho = raiz / nome
    if caminho.is_file() and not caminho.is_symlink():
        ficheiros_pacote.append(caminho)
ficheiros_pacote = sorted(set(ficheiros_pacote))
registo = {
    "produto": "Análise histórica da cobertura de médico de família — SNS",
    "pagina": ficheiro_html.relative_to(raiz).as_posix(),
    "periodo": resumo["periodo_principal"], "recolhido_em_utc": resumo["recolhido_em_utc"],
    "entrada_avaliacao_sha256": sha256(ficheiro_resumo), "inputs_sha256": assinaturas_antes,
    "selecionado_no_04": selecionado, "nova_modelacao_nesta_fase": False,
    "publicacao_externa": False, "formato": "Página local autónoma e pacote do projeto",
    "pendencias": resumo["pendencias"], "limites": resumo["limites"],
    "faltam_ficheiros_essenciais": faltam_pacote,
    "ficheiros": [{"caminho": p.relative_to(raiz).as_posix(), "bytes": p.stat().st_size, "sha256": sha256(p)} for p in ficheiros_pacote],
    "nota_assinaturas": "Lista exclui este manifesto e o próprio ZIP. O notebook 06 pode voltar a ser guardado após a execução; ao finalizar o pacote, executar esta célula novamente depois de guardar os notebooks.",
}
ficheiro_manifesto.write_text(json.dumps(registo, ensure_ascii=False, indent=2), encoding="utf-8")
if faltam_pacote:
    print("Não criar ZIP incompleto. Confirmar o inventário e voltar a executar esta célula.")
else:
    with zipfile.ZipFile(temporario_pacote, "w", compression=zipfile.ZIP_DEFLATED) as pacote:
        for p in ficheiros_pacote + [ficheiro_manifesto]:
            pacote.write(p, "Projeto_SNS/" + p.relative_to(raiz).as_posix())
    with zipfile.ZipFile(temporario_pacote) as pacote:
        if pacote.testzip() is not None:
            raise ValueError("A verificação do ZIP falhou.")
    temporario_pacote.replace(ficheiro_pacote)
    print("Pacote criado: data/output/Projeto_SNS.zip")
    print("Tamanho:", round(ficheiro_pacote.stat().st_size / (1024 * 1024), 2), "MB")
print("Disponibilização concluída. Página local e documentação prontas.")


Pacote criado: data/output/Projeto_SNS.zip
Tamanho: 4.18 MB
Disponibilização concluída. Página local e documentação prontas.


## Consultar o produto

[Abrir a página de resultados](../docs/06_resultados_sns.html)

[Consultar o README](../README.md)

[Abrir o pacote do projeto, depois de gerado](../data/output/Projeto_SNS.zip)

Abre a página num navegador se o VS Code apresentar apenas o código HTML. A página é autónoma e pode ser consultada fora da pasta do projeto.

## Fecho desta versão

Ficam disponíveis a análise histórica, os dados preparados, a comparação dos modelos e as decisões. O critério preditivo é apresentado com o resultado da avaliação, mesmo quando a referência simples ganha.

Continuam pendentes a correspondência territorial da Póvoa/Vila do Conde e a janela exata das taxas de consultas. Uma futura análise do indicador novo exige novo enquadramento e avaliação; a descontinuação da série antiga não é resolvida por prolongar a previsão.

A seguir, rever o README existente, o `.gitignore` e o pacote final. Esta fase prepara os ficheiros; não publica o projeto num serviço externo.